# Generate Parameter Ensembles
This script can create FATES parameter ensembles for a min/max one-at-a-time (OAAT) or latin hypercube (LH) experiments.

The main information required are a default FATES parameter file, an excel file with information about any parameters to be calibrated, and a list of parameters to include in the ensemble. 



In [52]:
import os
import pandas as pd
import xarray as xr
import numpy as np
import fates_calibration_library.parameter_generation as param

In [4]:
# top directory
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

# default parameter file
fates_param_name = "fates_params_default_sci.1.85.1_api.40.0.0_crops.nc"
# fates_param_name = "fates_params_default_sci.1.81.1_api.38.0.0_crops_vai_CLM_Medlyn_btran.nc"

# excel file with information about parameters
param_list_name = "param_list_sci.1.85.1_api.40.0.0_updates.xls"

# list of parameters to include in OAAT ensemble
oaat_params_file = 'oaat_params.csv'

# list of parameters to include in LH ensemble
lh_params_file = 'lh_params.csv'

# directory to place OAAT files
oaat_dir = os.path.join(param_dir, 'fates_oaat_3')

# directory to place LH files
lh_dir = os.path.join(param_dir, 'fates_lh')

In [5]:
# get files
param_list_file = os.path.join(param_dir, param_list_name)
default_param_data = xr.open_dataset(os.path.join(param_dir, fates_param_name))
param_dat = param.get_param_dictionary(param_list_file)

In [ ]:
oaat = False
lh = False

In [ ]:
oaat_params = ['fates_leaf_vcmax25top', 'fates_leaf_stomatal_slope_medlyn', 'fates_leaf_stomatal_intercept']
param.create_oaat_param_ensemble(param_dat, oaat_params, default_param_data, oaat_dir, 'FATES_OAAT_3')

In [ ]:
if oaat:
    # get list of parameters for OAAT experiment
    oaat_params = pd.read_csv(os.path.join(param_dir, oaat_params_file))['fates_parameter_name'].values
    
    # oaat ensemble
    # param.create_oaat_param_ensemble(param_dat, oaat_params, default_param_data,
    #                                  oaat_dir, 'FATES_OAAT')
    # oaat_key = pd.read_csv(os.path.join(oaat_dir, 'fates_oaat_key.csv'))

In [ ]:
if lh:
    # get list of parameters for LH experiment
    lh_params = pd.read_csv(os.path.join(param_dir, lh_params_file))['fates_parameter_name'].values

    # lh ensemble
    param.create_lh_param_ensemble(lh_params, 500, default_param_data,
                                   param_dat, lh_dir, 'FATES_LH')
    # check the key just in case
    lh_key = pd.read_csv(os.path.join(lh_dir, 'fates_lh_key.csv'), index_col=0)

In [ ]:
lh_key = pd.read_csv(os.path.join(lh_dir, 'fates_lh_key.csv'), index_col=0)
params = lh_key.columns[:-1]

In [ ]:
main_param = param_dat["main"]

In [ ]:
all_normalized = {}
for parameter in params:
    
    if parameter == 'smpsc_delta':
        actual_param_name = param.PARAM_INFO[parameter]['actual_param']
    elif parameter == 'fates_stoich_nitr_1':
        actual_param_name = param.PARAM_INFO[parameter]['actual_param']
    else:
        actual_param_name = parameter

    if parameter == 'fates_stoich_nitr_1':
        default_value = default_param_data[actual_param_name].isel(fates_plant_organs=0)
    else: 
        default_value = default_param_data[actual_param_name]
        
    sub = main_param[main_param.fates_parameter_name == parameter]
    
    # get min and max parameter values
    change_str_min = str(sub["param_min"].values[0])
    change_str_max = str(sub["param_max"].values[0])
    
    min_value = param.get_param_value(
        change_str_min, default_value, param_dat, parameter, "param_min"
    )
    max_value = param.get_param_value(
        change_str_max, default_value, param_dat, parameter, "param_max"
    )
    normalized = param.normalize(default_value, min_value, max_value)
    if normalized.size == 1:
        norm_values = np.full(16, normalized.values)
    else:
        norm_values = normalized.values.flatten()
    all_normalized[parameter] = norm_values
norm_df = pd.DataFrame(all_normalized)
norm_df['pft'] = np.arange(1, 17)

In [ ]:
norm_df.to_csv('/glade/work/afoster/FATES_calibration/parameter_files/normalized_parameters.csv')

In [6]:
out_dir = '/glade/work/afoster/FATES_calibration/parameter_outputs'
update_df = pd.read_csv(os.path.join(out_dir, 'all_dom_pfts_update.csv'))

In [69]:
lh_dir = '/glade/work/afoster/FATES_calibration/parameter_files/fates_lh_codom'
lh_dom_dir = os.path.join(param_dir, 'fates_lh')

In [65]:
files = sorted([os.path.join(lh_dom_dir, f) for f in os.listdir(lh_dom_dir) if f.endswith('.nc')])
params = np.unique(update_df.parameter_name)

In [71]:
for file in files:
    dat = xr.open_dataset(file)
    new_param = dat.copy(deep=False)

    for parameter in params:
        sub = update_df[update_df.parameter_name == parameter].copy()
        
        if parameter in param.PARAM_INFO:
            parameter_name = param.PARAM_INFO[parameter]['actual_param']
            array_index = param.PARAM_INFO[parameter]['array_index']
            
            new_values = new_param[parameter_name].copy()
            for id, value in zip(sub['pft_id'], sub['parameter_value']):
                new_values[array_index, id-1] = value
        else:
            parameter_name = parameter
            
            new_values = new_param[parameter_name].copy()
            for id, value in zip(sub['pft_id'], sub['parameter_value']):
                new_values[id-1] = value
                
        new_param[parameter_name].values = new_values
    new_param.to_netcdf(os.path.join(lh_dir, os.path.basename(file)))